In [1]:
from test.helpers import SDE, simple_batch_sde_solve

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
from diffrax import ControlTerm, MultiTerm, ODETerm, SaveAt, SlowRK, SpaceTimeLevyArea


jax.config.update("jax_enable_x64", True)
dtype = jnp.float64


def drift(t, y, args):
    a, b, sigma = args
    return a * (b - y)


def diffusion(t, y, args):
    a, b, sigma = args
    return sigma * y


def get_terms(bm):
    return MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))


t0 = 0.0
t1 = 5.0
y0_flt = 0.16
y0 = jnp.array(y0_flt, dtype=dtype)
ito_args = (1.0, 0.1, 1.4)


def get_stratonovich_args(a: float, b: float, sigma: float):
    tilde_a = a + sigma**2 / 2
    tilde_b = a * b / tilde_a
    return (
        jnp.array(tilde_a, dtype=dtype),
        jnp.array(tilde_b, dtype=dtype),
        jnp.array(sigma, dtype=dtype),
    )


strat_args = get_stratonovich_args(*ito_args)
w_shape = ()
igbm = SDE(get_terms, strat_args, y0, t0, t1, w_shape)

In [2]:
GLOBAL_KEY = jr.PRNGKey(9)

num_samples = 2**15
dt0 = 2**-10
bm_tol = 0.25 * dt0
saveat = SaveAt(t1=True)
y1s_list = []
for i in range(64):
    GLOBAL_KEY, temp_key = jr.split(GLOBAL_KEY, 2)
    keys = jr.split(temp_key, num_samples)
    y1, _ = simple_batch_sde_solve(
        keys, igbm, SlowRK(), SpaceTimeLevyArea, dt0, None, bm_tol, saveat
    )
    y1 = jnp.squeeze(y1)
    y1s_list.append(y1)
    print(f"done {i}")
y1s = jnp.concatenate(y1s_list)
print(y1s.shape)

done 0
done 1
done 2
done 3
done 4
done 5
done 6
done 7
done 8
done 9
done 10
done 11
done 12
done 13
done 14
done 15
done 16
done 17
done 18
done 19
done 20
done 21
done 22
done 23
done 24
done 25
done 26
done 27
done 28
done 29
done 30
done 31
done 32
done 33
done 34
done 35
done 36
done 37
done 38
done 39
done 40
done 41
done 42
done 43
done 44
done 45
done 46
done 47
done 48
done 49
done 50
done 51
done 52
done 53
done 54
done 55
done 56
done 57
done 58
done 59
done 60
done 61
done 62
done 63
(2097152,)


In [4]:
ito_args_str = "_".join([str(arg) for arg in ito_args])
print(ito_args_str)
file_name = (
    f"/home/andy/PycharmProjects/Levy_CFGAN/jax_src/sst"
    f"/igbm_{y0_flt}_{ito_args_str}_SlowRK_2^21.npy"
)
with open(file_name, "wb") as f:
    np.save(f, np.array(y1s))

1.0_0.1_1.4
